In [112]:
import pandas as pd
import geopandas as gpd
import time
import maup
from maup import smart_repair

In [113]:
start_time = time.time()
election_df = gpd.read_file(r".\pa_2024_gen_prec_draft\pa_2024_gen_prec_draft.shp")
end_time = time.time()

print("The time to import pa_2024_gen_prec_draft.shp is:", (end_time-start_time)/60, "mins")

The time to import pa_2024_gen_prec_draft.shp is: 0.0224381685256958 mins


c:\Users\dcviv\miniconda3\envs\vini\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: .\pa_2024_gen_prec_draft\pa_2024_gen_prec_draft.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


In [114]:
print(election_df.columns)

Index(['UNIQUE_ID', 'COUNTYFP', 'County', 'Precinct', 'Muni_Name', 'Muni_Type',
       'Muni_Code1', 'Muni_Name1', 'Muni_Code2', 'Muni_Name2',
       ...
       'GSU41RPIT', 'GSU43DCOS', 'GSU45DPIS', 'GSU45RDIN', 'GSU47DLEN',
       'GSU47RVOG', 'GSU49DWER', 'GSU49RLAU', 'TOT_VOTES', 'geometry'],
      dtype='object', length=429)


In [115]:
start_time = time.time()
population_df = gpd.read_file(r".\pa_pl2020_b\pa_pl2020_p2_b.shp")
end_time = time.time()

print("The time to import pa_pl2020_p2_b.shp is:", (end_time-start_time)/60, "mins")

The time to import pa_pl2020_p2_b.shp is: 0.16871732473373413 mins


In [116]:
population_df.columns

Index(['GEOID20', 'SUMLEV', 'LOGRECNO', 'GEOID', 'COUNTY', 'P0020001',
       'P0020002', 'P0020003', 'P0020004', 'P0020005', 'P0020006', 'P0020007',
       'P0020008', 'P0020009', 'P0020010', 'P0020011', 'P0020012', 'P0020013',
       'P0020014', 'P0020015', 'P0020016', 'P0020017', 'P0020018', 'P0020019',
       'P0020020', 'P0020021', 'P0020022', 'P0020023', 'P0020024', 'P0020025',
       'P0020026', 'P0020027', 'P0020028', 'P0020029', 'P0020030', 'P0020031',
       'P0020032', 'P0020033', 'P0020034', 'P0020035', 'P0020036', 'P0020037',
       'P0020038', 'P0020039', 'P0020040', 'P0020041', 'P0020042', 'P0020043',
       'P0020044', 'P0020045', 'P0020046', 'P0020047', 'P0020048', 'P0020049',
       'P0020050', 'P0020051', 'P0020052', 'P0020053', 'P0020054', 'P0020055',
       'P0020056', 'P0020057', 'P0020058', 'P0020059', 'P0020060', 'P0020061',
       'P0020062', 'P0020063', 'P0020064', 'P0020065', 'P0020066', 'P0020067',
       'P0020068', 'P0020069', 'P0020070', 'P0020071', 'P002

In [117]:
start_time = time.time()
county_df = population_df.dissolve(by=['COUNTY'])
end_time = time.time()

print("The time to import county data from population_df is:", (end_time-start_time)/60, "mins")

The time to import county data from population_df is: 0.924144697189331 mins


In [118]:
county_df.columns

Index(['geometry', 'GEOID20', 'SUMLEV', 'LOGRECNO', 'GEOID', 'P0020001',
       'P0020002', 'P0020003', 'P0020004', 'P0020005', 'P0020006', 'P0020007',
       'P0020008', 'P0020009', 'P0020010', 'P0020011', 'P0020012', 'P0020013',
       'P0020014', 'P0020015', 'P0020016', 'P0020017', 'P0020018', 'P0020019',
       'P0020020', 'P0020021', 'P0020022', 'P0020023', 'P0020024', 'P0020025',
       'P0020026', 'P0020027', 'P0020028', 'P0020029', 'P0020030', 'P0020031',
       'P0020032', 'P0020033', 'P0020034', 'P0020035', 'P0020036', 'P0020037',
       'P0020038', 'P0020039', 'P0020040', 'P0020041', 'P0020042', 'P0020043',
       'P0020044', 'P0020045', 'P0020046', 'P0020047', 'P0020048', 'P0020049',
       'P0020050', 'P0020051', 'P0020052', 'P0020053', 'P0020054', 'P0020055',
       'P0020056', 'P0020057', 'P0020058', 'P0020059', 'P0020060', 'P0020061',
       'P0020062', 'P0020063', 'P0020064', 'P0020065', 'P0020066', 'P0020067',
       'P0020068', 'P0020069', 'P0020070', 'P0020071', 'P0

In [119]:
start_time = time.time()
district_df = gpd.read_file(r".\pa_cong_adopted_2022\carter_boundaries.shp")
end_time = time.time()

print("The time to import carter_boundaries is:", (end_time-start_time)/60, "mins")

The time to import carter_boundaries is: 0.0006834149360656738 mins


In [120]:
district_df.columns

Index(['ID', 'AREA', 'DISTRICT', 'geometry'], dtype='object')

The 2024 election_df covers a lot of elections and county info but doesn't have population information and district assignments so I need to merge PL 94-171 census data and assign districts

In [121]:
# Convert to UTM
population_df = population_df.to_crs(population_df.estimate_utm_crs())
election_df = election_df.to_crs(election_df.estimate_utm_crs())
district_df = district_df.to_crs(district_df.estimate_utm_crs())
county_df = county_df.to_crs(county_df.estimate_utm_crs())

In [122]:
# maup.doctor raised a topology error on the precinct shapefile.
# I then attempted a full smart_repair pass, but it was computationally expensive
# on this dataset and did not finish in a practical amount of time.
election_df = smart_repair(election_df)

Snapping all geometries to a grid with precision 10^( -4 ) to avoid GEOS errors.
Identifying overlaps...
Resolving overlaps...
Assigning order 2 pieces...
Assigning order 3 pieces...
Couldn't find a polygon to glue a component in the intersection of geometries {1154, 51, 1156} to
Couldn't find a polygon to glue a component in the intersection of geometries {330, 331, 332} to
Couldn't find a polygon to glue a component in the intersection of geometries {938, 515, 516} to
1 gaps will remain unfilled, because they exceed the area threshold.
1 gaps will remain unfilled, because they are not simply connected.
Filling gaps...


Gaps to simplify: 6115it [3:56:15,  2.32s/it]                             
Gaps to fill: 100%|██████████| 819/819 [57:35<00:00,  4.22s/it]  


In [123]:
maup.doctor(county_df)

True

In [124]:
# Check for issues
maup.doctor(election_df)

There are 5 holes.


False

In [125]:
# Check for issues
maup.doctor(population_df)

True

In [126]:
# Check for issues
maup.doctor(district_df)

True

In [127]:
# county_regions = election_df.dissolve(by="County")

election_df = smart_repair(
    election_df,
    # nest_within_regions=county_df,
    min_rook_length=30
)

Snapping all geometries to a grid with precision 10^( -4 ) to avoid GEOS errors.
Identifying overlaps...
Resolving overlaps...
1 gaps will remain unfilled, because they exceed the area threshold.
1 gaps will remain unfilled, because they are not simply connected.
Filling gaps...


Gaps to simplify: 100%|██████████| 3/3 [00:06<00:00,  2.25s/it]
Gaps to fill: 0it [00:00, ?it/s]


Converting small rook adjacencies to queen...


c:\Users\dcviv\miniconda3\envs\vini\Lib\site-packages\maup\adjacencies.py:91: IslandWarning: Found islands.
Indices of islands: {3940, 3998, 3861, 629, 3964, 3869, 3870, 3999}
  warnings.warn(


try this first instead of regular smart repair()

In [128]:
maup.doctor(election_df)

There are 2 holes.


False

In [129]:
# maup.doctor(repaired_election_county)

In [130]:
election_df.to_file("pa_2024_gen_prec_repaired_county.shp")

c:\Users\dcviv\miniconda3\envs\vini\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Creating a 256th field, but some DBF readers might only support 255 fields
  ogr_write(


In [131]:
final_df = election_df.copy()

In [132]:
# Assign each census block to a precinct
blocks_to_precinct = maup.assign(population_df, final_df.geometry)

c:\Users\dcviv\miniconda3\envs\vini\Lib\site-packages\maup\assign.py:32: AssigmentWarning: Warning: Some units in the source geometry were unassigned.
  warnings.warn(


In [133]:
blocks_to_precinct

0         2997.0
1         6282.0
2         6283.0
3         6263.0
4         6273.0
           ...  
336980    1963.0
336981    1912.0
336982    1949.0
336983    1969.0
336984    1961.0
Length: 336985, dtype: float64

In [134]:
pop_columns = ['P0020001', 'P0020002', 'P0020005', 'P0020006', 'P0020007', 'P0020008', 'P0020009', 'P0020010', 'P0020011']

In [135]:
# Sum the block population into each precinct
for name in pop_columns:
    final_df[name] = population_df[name].groupby(blocks_to_precinct).sum()

In [136]:
final_df.columns

Index(['UNIQUE_ID', 'COUNTYFP', 'County', 'Precinct', 'Muni_Name', 'Muni_Type',
       'Muni_Code1', 'Muni_Name1', 'Muni_Code2', 'Muni_Name2',
       ...
       'geometry', 'P0020001', 'P0020002', 'P0020005', 'P0020006', 'P0020007',
       'P0020008', 'P0020009', 'P0020010', 'P0020011'],
      dtype='object', length=438)

In [137]:
# Check that no one was lost
print(population_df['P0020001'].sum())
print(final_df['P0020001'].sum())

13002700
13002674.0


prof is okay with losing 30 people. Mention it!!

In [138]:
rename = {'P0020001': 'TOTPOP', 'P0020002': 'HISP', 'P0020005': 'NH_WHITE', 'P0020006': 'NH_BLACK', 'P0020007': 'NH_AMIN', 'P0020008': 'NH_ASIAN', 'P0020009': 'NH_NHPI', 'P0020010': 'NH_OTHER', 'P0020011': 'NH_2MORE'}

In [139]:
# Rename the column names
final_df.rename(columns = rename, inplace = True)

In [140]:
final_df.columns

Index(['UNIQUE_ID', 'COUNTYFP', 'County', 'Precinct', 'Muni_Name', 'Muni_Type',
       'Muni_Code1', 'Muni_Name1', 'Muni_Code2', 'Muni_Name2',
       ...
       'geometry', 'TOTPOP', 'HISP', 'NH_WHITE', 'NH_BLACK', 'NH_AMIN',
       'NH_ASIAN', 'NH_NHPI', 'NH_OTHER', 'NH_2MORE'],
      dtype='object', length=438)